In [ ]:
import sys, os, json, warnings, time as _time
import tempfile

# GPU 选择：修改这里指定使用哪张 GPU（物理编号）
GPU_ID = 5
os.environ['CUDA_VISIBLE_DEVICES'] = str(GPU_ID)

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.linalg import sqrtm

sys.path.insert(0, '/data/yc/Fluid_VSR')
warnings.filterwarnings('ignore')

from forecastors import BaseForecaster, ResshiftForecaster, RemgForecaster
from datasets import _dataset_dict

print(f'Imports OK. Using physical GPU {GPU_ID} (mapped to cuda:0)')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 唯一需要修改的地方（GPU 选择在第一个 cell 的 GPU_ID）
# ══════════════════════════════════════════════════════════════════════════════

# DATASET_NAME = 'KF256'   # 'RB' / 'KF256' / 'ShallowWater' / 'ERA5V' 等
# DATASET_NAME = 'RB'   # 'RB' / 'KF256' / 'ShallowWater' / 'ERA5V' 等
# DATASET_NAME = 'ShallowWater'   # 'RB' / 'KF256' / 'ShallowWater' / 'ERA5V' 等
DATASET_NAME = 'ERA5V'   # 'RB' / 'KF256' / 'ShallowWater' / 'ERA5V' 等

device = 'cuda:0'  # GPU_ID 已在 imports cell 通过 CUDA_VISIBLE_DEVICES 映射，始终用 cuda:0

# 各数据集的帧数（用于自动适配 vis_frames）
_FRAMES_PER_SEQ = {'RB': 100, 'KF256': 180, 'ShallowWater': 72, 'ERA5V': 24}
_total_frames = _FRAMES_PER_SEQ.get(DATASET_NAME, 100)

# 是否计算涡度/拟涡能指标（仅 KF256）
include_vorticity = (DATASET_NAME == 'KF256')

# 可视化帧索引（自动适配，均匀分布 5 帧）
_step = max(1, _total_frames // 5)
vis_frames = list(range(_step - 1, _total_frames, _step))[:5]

# 图片保存根目录
output_image_dir = f'/data/yc/output_images/{DATASET_NAME}'

models = [
    {
        'name': 'FNO',
        'type': 'fluid_vsr',
        'forecaster': 'base',
        # 'model_dir': '/data/yc/Fluid_VSR/logs/KolmogorovFlow/FNO/03_16/FNO2d_11_43_19',
        # 'model_dir': '/data/yc/Fluid_VSR/logs/RayleighBenard/FNO2d_10_07_25',
        # 'model_dir': '/data/yc/Fluid_VSR/logs/ShallowWater/FNO/03_19/FNO2d_12_46_41',
        'model_dir': '/data/yc/Fluid_VSR/logs/ERA5V/FNO/03_25/FNO2d_07_25_39',
    },
    {
        'name': 'EDSR',
        'type': 'fluid_vsr',
        'forecaster': 'base',
        # 'model_dir': '/data/yc/Fluid_VSR/logs/KolmogorovFlow/EDSR_11_47_12',
        # 'model_dir': '/data/yc/Fluid_VSR/logs/RayleighBenard/EDSR_10_19_25',
        # 'model_dir': '/data/yc/Fluid_VSR/logs/ShallowWater/EDSR/03_19/EDSR_12_49_24',
        'model_dir': '/data/yc/Fluid_VSR/logs/ERA5V/EDSR/03_25/EDSR_07_27_12',
    },
    {
        'name': 'SRNO',
        'type': 'fluid_vsr',
        'forecaster': 'base',
        # 'model_dir': '/data/yc/Fluid_VSR/logs/KolmogorovFlow/SRNO/03_16/SRNO_11_51_13',
        # 'model_dir': '/data/yc/Fluid_VSR/logs/RayleighBenard/SRNO_09_35_50',
        # 'model_dir': '/data/yc/Fluid_VSR/logs/ShallowWater/SRNO/03_19/SRNO_12_52_43',
        'model_dir': '/data/yc/Fluid_VSR/logs/ERA5V/SRNO/03_25/SRNO_07_28_38',
    },
    {
        'name': 'SwinIR',
        'type': 'fluid_vsr',
        'forecaster': 'base',
        # 'model_dir': '/data/yc/Fluid_VSR/logs/KolmogorovFlow/SwinIR/03_16/SwinIR_11_54_35',
        # 'model_dir': '/data/yc/Fluid_VSR/logs/RayleighBenard/SwinIR_14_19_20',
        # 'model_dir': '/data/yc/Fluid_VSR/logs/ShallowWater/SwinIR/03_19/SwinIR_12_54_38',
        'model_dir': '/data/yc/Fluid_VSR/logs/ERA5V/SwinIR/03_25/SwinIR_07_30_53',
    },
    # {
    #     'name': 'ResShift',
    #     'type': 'fluid_vsr',
    #     'forecaster': 'resshift',
    #     # 'model_dir': '/data/yc/Fluid_VSR/logs/KolmogorovFlow/Resshift/03_16/Resshift_14_35_00',
    #     'model_dir': '/data/yc/Fluid_VSR/logs/RayleighBenard/Resshift/03_13/Resshift_09_27_12',
    #     # 'model_dir': '/data/yc/Fluid_VSR/logs/ERA5V/Resshift/...',
    # },
    # {
    #     'name': 'ReMD',
    #     'type': 'fluid_vsr',
    #     'forecaster': 'remg',
    #     # 'model_dir': '/data/yc/Fluid_VSR/logs/KolmogorovFlow/ReMG/03_16/ReMG_14_43_27',
    #     'model_dir': '/data/yc/Fluid_VSR/logs/RayleighBenard/ReMG/03_13/ReMG_10_00_04',
    #     # 'model_dir': '/data/yc/Fluid_VSR/logs/ERA5V/ReMG/...',
    # },
    # {
    #     'name': 'VRT',
    #     'type': 'external',
    #     'pred_npz':  '/data/yc/Fluid_VSR/vsr_baseline/rb_dataset/vrt_pred/pred.npz',
    #     'gt_npz':    '/data/yc/Fluid_VSR/vsr_baseline/rb_dataset/vrt_pred/gt.npz',
    #     'lr_npz':    '/data/yc/Fluid_VSR/vsr_baseline/rb_dataset/vrt_pred/lr.npz',
    #     'meta_json': '/data/yc/Fluid_VSR/vsr_baseline/rb_dataset/vrt_pred/meta.json',
    # },
    # {
    #     'name': 'RVRT',
    #     'type': 'external',
    #     'pred_npz':  '/data/yc/Fluid_VSR/vsr_baseline/rb_dataset/rvrt_pred/pred.npz',
    #     'gt_npz':    '/data/yc/Fluid_VSR/vsr_baseline/rb_dataset/rvrt_pred/gt.npz',
    #     'lr_npz':    '/data/yc/Fluid_VSR/vsr_baseline/rb_dataset/rvrt_pred/lr.npz',
    #     'meta_json': '/data/yc/Fluid_VSR/vsr_baseline/rb_dataset/rvrt_pred/meta.json',
    # },
    # {
    #     'name': 'BasicVSR++',
    #     'type': 'external',
    #     'pred_npz':  '/data/yc/Fluid_VSR/vsr_baseline/rb_dataset/basicvsr++_pred/pred.npz',
    #     'gt_npz':    '/data/yc/Fluid_VSR/vsr_baseline/rb_dataset/basicvsr++_pred/gt.npz',
    #     'lr_npz':    '/data/yc/Fluid_VSR/vsr_baseline/rb_dataset/basicvsr++_pred/lr.npz',
    #     'meta_json': '/data/yc/Fluid_VSR/vsr_baseline/rb_dataset/basicvsr++_pred/meta.json',
    # },
    # {
    #     'name': 'SDIFT',
    #     'type': 'external',
    #     'pred_npz':  '/data/yc/Fluid_VSR/vsr_baseline/sw_dataset/sdift/pred.npz',
    #     'gt_npz':    '/data/yc/Fluid_VSR/vsr_baseline/sw_dataset/sdift/gt.npz',
    #     'lr_npz':    '/data/yc/Fluid_VSR/vsr_baseline/sw_dataset/sdift/lr.npz',
    #     'meta_json': '/data/yc/Fluid_VSR/vsr_baseline/sw_dataset/sdift/meta.json',
    # },
]

SDIFT_NAME = 'SDIFT'

metrics    = ['basic', 'psdd', 'fid', 'fvd']  # 可选: 'basic' 'psdd' 'fid' 'fvd'
batch_size = 64

vis_seq_id = 0
err_colorbar_percentile = 98

_efficiency_name_map = {'FNO': 'FNO2d', 'ReMD': 'REMG'}

print(f'Config OK. DATASET_NAME={DATASET_NAME}, device={device} (physical GPU {GPU_ID})')
print(f'  include_vorticity={include_vorticity}, vis_frames={vis_frames}')
print(f'  output_image_dir={output_image_dir}')
for m in models:
    print(f"  {m['name']} ({m['type']})")

In [ ]:
# ── PSDD: Power Spectrum Density Discrepancy ──────────────────────────────────
def compute_psdd(pred, gt, eps=1e-12):
    """
    pred, gt: (N, H, W, C) float32 numpy arrays (physical values).
    Returns scalar PSDD (lower = better spectral agreement).
    """
    x = pred.astype(np.float64) - pred.mean(axis=(1, 2), keepdims=True)
    y = gt.astype(np.float64)   - gt.mean(axis=(1, 2), keepdims=True)

    Fx = np.fft.fftshift(np.fft.fft2(x, axes=(1, 2)), axes=(1, 2))
    Fy = np.fft.fftshift(np.fft.fft2(y, axes=(1, 2)), axes=(1, 2))

    def norm_psd(F):
        s = np.maximum(
            np.abs(F.real).max(axis=(1, 2, 3), keepdims=True),
            np.abs(F.imag).max(axis=(1, 2, 3), keepdims=True)
        )
        s = np.maximum(s, eps)
        P = (F.real / s) ** 2 + (F.imag / s) ** 2
        return P / (P.sum(axis=(1, 2, 3), keepdims=True) + eps)

    Px = norm_psd(Fx)
    Py = norm_psd(Fy)
    return float(np.abs(Px - Py).mean(axis=(1, 2, 3)).mean())

In [ ]:

# ── 对每个模型：推理（或加载缓存） + 计算指标 ─────────────────────────────────

import time as _time
from torchmetrics.functional.image import (
    peak_signal_noise_ratio,
    structural_similarity_index_measure,
)

all_results = {}   # name -> {metric: value}
all_arrays  = {}   # name -> {'pred', 'gt', 'lr', 'frames_per_seq'}
all_efficiency = {}  # name -> {'infer_time_total', 'infer_time_per_seq', 'peak_memory_mb'}

# ── Forecaster 工厂 ───────────────────────────────────────────────────────────
_FORECASTER_MAP = {
    'base':     BaseForecaster,
    'resshift': ResshiftForecaster,
    'remg':     RemgForecaster,
}

def get_forecaster(m):
    cls = _FORECASTER_MAP.get(m.get('forecaster', 'base'), BaseForecaster)
    fc = cls(m['model_dir'])
    # 覆盖 forecaster 的 device 为 config 中指定的 GPU
    fc.device = device
    fc.model.to(device)
    return fc


def run_inference_fluid_vsr(forecaster, dataset, output_dir):
    """Run inference on test set with timing. Returns (pred_arr, gt_arr, lr_arr, efficiency_dict)."""
    normalizer     = dataset.normalizer
    frames_per_seq = dataset.frames_per_seq
    eval_bs = forecaster.data_args.get('eval_batchsize', 16)
    test_loader = torch.utils.data.DataLoader(
        dataset.test_dataset, batch_size=eval_bs,
        shuffle=False, num_workers=0, pin_memory=True)

    n_seqs = len(dataset.test_dataset) // frames_per_seq if frames_per_seq else 1

    # Warmup (3 batches)
    print('  Warmup (3 batches)...')
    forecaster.model.eval()
    with torch.no_grad():
        for wi, (x, y) in enumerate(test_loader):
            if wi >= 3:
                break
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            _ = forecaster.inference(x, y)

    # Reset peak memory and start timing
    if device.startswith('cuda'):
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

    all_pred, all_gt, all_lr = [], [], []
    t_start = _time.time()

    with torch.no_grad():
        for i, (x, y) in enumerate(test_loader):
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            y_pred = forecaster.inference(x, y)
            all_pred.append(normalizer.decode(y_pred).cpu().numpy())
            all_gt.append(normalizer.decode(y).cpu().numpy())
            all_lr.append(dataset.lr_normalizer.decode(x).cpu().numpy())
            if (i + 1) % 20 == 0:
                print(f'    batch {i+1}/{len(test_loader)}')

    if device.startswith('cuda'):
        torch.cuda.synchronize()
    t_total = _time.time() - t_start

    # Peak memory
    peak_mem_mb = None
    if device.startswith('cuda'):
        peak_mem_mb = torch.cuda.max_memory_allocated() / (1024 * 1024)

    pred_arr = np.concatenate(all_pred, axis=0)
    gt_arr   = np.concatenate(all_gt,   axis=0)
    lr_arr   = np.concatenate(all_lr,   axis=0)

    np.savez_compressed(os.path.join(output_dir, 'pred.npz'), data=pred_arr)
    np.savez_compressed(os.path.join(output_dir, 'gt.npz'),   data=gt_arr)
    np.savez_compressed(os.path.join(output_dir, 'lr.npz'),   data=lr_arr)
    meta = {
        'n_frames': int(pred_arr.shape[0]),
        'frames_per_seq': frames_per_seq,
        'n_seqs': int(pred_arr.shape[0]) // frames_per_seq,
        'pred_shape': list(pred_arr.shape),
        'lr_note': 'LR in normalized (model input) space',
    }
    with open(os.path.join(output_dir, 'meta.json'), 'w') as f:
        json.dump(meta, f, indent=2)

    eff = {
        'infer_time_total': round(t_total, 2),
        'infer_time_per_seq': round(t_total / max(n_seqs, 1), 4),
        'n_seqs': n_seqs,
        'peak_memory_mb': round(peak_mem_mb, 2) if peak_mem_mb else None,
    }
    print(f'  Inference: {t_total:.2f}s total, {eff["infer_time_per_seq"]:.4f}s/seq, '
          f'peak mem: {peak_mem_mb:.0f}MB' if peak_mem_mb else '')

    return pred_arr, gt_arr, lr_arr, eff


def run_inference_cached(forecaster, dataset, output_dir):
    """For cached predictions, still measure inference time by re-running."""
    frames_per_seq = dataset.frames_per_seq
    eval_bs = forecaster.data_args.get('eval_batchsize', 16)
    test_loader = torch.utils.data.DataLoader(
        dataset.test_dataset, batch_size=eval_bs,
        shuffle=False, num_workers=0, pin_memory=True)

    n_seqs = len(dataset.test_dataset) // frames_per_seq if frames_per_seq else 1

    # Warmup
    forecaster.model.eval()
    with torch.no_grad():
        for wi, (x, y) in enumerate(test_loader):
            if wi >= 3:
                break
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            _ = forecaster.inference(x, y)

    if device.startswith('cuda'):
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

    t_start = _time.time()
    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            _ = forecaster.inference(x, y)

    if device.startswith('cuda'):
        torch.cuda.synchronize()
    t_total = _time.time() - t_start

    peak_mem_mb = None
    if device.startswith('cuda'):
        peak_mem_mb = torch.cuda.max_memory_allocated() / (1024 * 1024)

    return {
        'infer_time_total': round(t_total, 2),
        'infer_time_per_seq': round(t_total / max(n_seqs, 1), 4),
        'n_seqs': n_seqs,
        'peak_memory_mb': round(peak_mem_mb, 2) if peak_mem_mb else None,
    }


def compute_basic_metrics(pred_arr, gt_arr, dev, batch_size=256, include_vorticity=True):
    mse_acc = psnr_acc = ssim_acc = rl2_acc = total = 0
    ve_list, ee_list = [], []
    for i in range(0, len(pred_arr), batch_size):
        p = torch.tensor(pred_arr[i:i+batch_size]).permute(0,3,1,2).to(dev)
        g = torch.tensor(gt_arr[i:i+batch_size]).permute(0,3,1,2).to(dev)
        b = p.shape[0]
        dr = float(g.max() - g.min())
        mse_acc  += F.mse_loss(p, g).item() * b
        psnr_acc += peak_signal_noise_ratio(p, g, data_range=dr).item() * b
        ssim_acc += structural_similarity_index_measure(p, g, data_range=dr).item() * b
        rl2_acc  += ((p-g).pow(2).sum((1,2,3)).sqrt() /
                      g.pow(2).sum((1,2,3)).sqrt().clamp_min(1e-12)).mean().item() * b
        if include_vorticity:
            spatial_mse = torch.mean((p - g) ** 2, dim=[1, 2, 3])
            ve_list.append(spatial_mse.cpu().numpy())
            enstrophy_pred = torch.mean(p ** 2, dim=[1, 2, 3]) / 2.0
            enstrophy_gt   = torch.mean(g ** 2, dim=[1, 2, 3]) / 2.0
            ee_list.append(((enstrophy_pred - enstrophy_gt) ** 2).cpu().numpy())
        total += b
    mse = mse_acc / total
    results = {'mse': mse, 'rmse': mse**0.5,
               'psnr': psnr_acc/total, 'ssim': ssim_acc/total,
               'relative_l2': rl2_acc/total}
    if include_vorticity:
        results['vorticity_error'] = float(np.sqrt(np.concatenate(ve_list).mean()))
        results['enstrophy_error'] = float(np.concatenate(ee_list).mean())
    return results


# ── 预先加载一次 dataset（用第一个 fluid_vsr 模型的路径）──────────────────────
_first_fluid = next((m for m in models if m['type'] == 'fluid_vsr'), None)
if _first_fluid:
    _ref_fc = get_forecaster(_first_fluid)
    _shared_dataset = _dataset_dict[_ref_fc.data_args['name']](_ref_fc.data_args)
    print(f'Dataset loaded (frames_per_seq={_shared_dataset.frames_per_seq})')
    del _ref_fc

# ── 主循环 ─────────────────────────────────────────────────────────────────────
for m in models:
    name = m['name']
    print(f'\n{"="*55}\n  {name}\n{"="*55}')

    # ── 加载 / 推理 ──────────────────────────────────────────────────────────
    if m['type'] == 'fluid_vsr':
        model_dir  = m['model_dir']
        output_dir = os.path.join(model_dir, 'test_predictions')
        os.makedirs(output_dir, exist_ok=True)

        forecaster     = get_forecaster(m)
        dataset        = _shared_dataset
        frames_per_seq = dataset.frames_per_seq

        pred_path = os.path.join(output_dir, 'pred.npz')
        if os.path.exists(pred_path):
            print('  加载缓存...')
            pred_arr = np.load(pred_path)['data']
            gt_arr   = np.load(os.path.join(output_dir, 'gt.npz'))['data']
            lr_path  = os.path.join(output_dir, 'lr.npz')
            if os.path.exists(lr_path):
                lr_arr = np.load(lr_path)['data']
            else:
                print('  从 dataset 加载 LR...')
                test_loader = torch.utils.data.DataLoader(
                    dataset.test_dataset, batch_size=16, shuffle=False, num_workers=0)
                all_lr = []
                with torch.no_grad():
                    for x, _ in test_loader:
                        all_lr.append(dataset.lr_normalizer.decode(x).cpu().numpy())
                lr_arr = np.concatenate(all_lr, axis=0)
                np.savez_compressed(lr_path, data=lr_arr)

            # 即使有缓存，也重新跑一遍推理来测时间
            print('  测量推理效率...')
            eff = run_inference_cached(forecaster, dataset, output_dir)
            all_efficiency[name] = eff
            print(f'  Inference: {eff["infer_time_total"]:.2f}s total, '
                  f'{eff["infer_time_per_seq"]:.4f}s/seq, '
                  f'peak mem: {eff["peak_memory_mb"]:.0f}MB' if eff["peak_memory_mb"] else '')
        else:
            print('  推理中...')
            pred_arr, gt_arr, lr_arr, eff = run_inference_fluid_vsr(
                forecaster, dataset, output_dir)
            all_efficiency[name] = eff

    elif m['type'] == 'external':
        print('  加载外部 npz...')
        pred_arr = np.load(m['pred_npz'])['data']
        gt_arr   = np.load(m['gt_npz'])['data']
        lr_arr   = np.load(m['lr_npz'])['data'] if 'lr_npz' in m else None
        if 'meta_json' in m and os.path.exists(m['meta_json']):
            with open(m['meta_json']) as f:
                frames_per_seq = json.load(f).get('frames_per_seq', 100)
        else:
            frames_per_seq = m.get('frames_per_seq', 100)
    else:
        raise ValueError(f"Unknown model type: {m['type']}")

    all_arrays[name] = {
        'pred': pred_arr, 'gt': gt_arr, 'lr': lr_arr,
        'frames_per_seq': frames_per_seq,
    }
    print(f'  pred: {pred_arr.shape}, gt: {gt_arr.shape}')

    # ── 指标 ─────────────────────────────────────────────────────────────────
    results = {}

    if 'basic' in metrics:
        print('  Computing basic metrics (+ vorticity)...')
        results.update(compute_basic_metrics(pred_arr, gt_arr, device, include_vorticity=True))

    if 'psdd' in metrics:
        print('  Computing PSDD...')
        results['psdd'] = compute_psdd(pred_arr, gt_arr)

    if 'fid' in metrics:
        print('  Computing FID...')
        try:
            import tempfile
            from cleanfid import fid as cleanfid
            from PIL import Image
            vmin = float(min(pred_arr.min(), gt_arr.min()))
            vmax = float(max(pred_arr.max(), gt_arr.max()))
            def save_png(arr, d):
                for i, f in enumerate(arr):
                    img = ((f.squeeze()-vmin)/(vmax-vmin+1e-12)*255).clip(0,255).astype(np.uint8)
                    Image.fromarray(np.stack([img]*3,-1)).save(os.path.join(d,f'{i:07d}.png'))
            with tempfile.TemporaryDirectory() as pd_, tempfile.TemporaryDirectory() as gd_:
                save_png(pred_arr, pd_); save_png(gt_arr, gd_)
                results['fid'] = cleanfid.compute_fid(
                    gd_, pd_, device=device, batch_size=batch_size, verbose=False)
        except ImportError:
            print('    跳过 FID: pip install clean-fid')

    if 'fvd' in metrics:
        print('  Computing FVD (R3D-18)...')
        try:
            import torchvision.models.video as vm
            from scipy.linalg import sqrtm
            _r3d = vm.r3d_18(weights=vm.R3D_18_Weights.KINETICS400_V1)
            r3d = torch.nn.Sequential(
                _r3d.stem,_r3d.layer1,_r3d.layer2,_r3d.layer3,_r3d.layer4,
                torch.nn.AdaptiveAvgPool3d(1)).eval().to(device)
            _mu = torch.tensor([0.43216,0.394666,0.37645]).view(1,3,1,1,1)
            _sd = torch.tensor([0.22803,0.22145,0.216989]).view(1,3,1,1,1)
            def r3d_feats(seqs):
                vn,vx = seqs.min(), seqs.max()
                s = (seqs-vn)/(vx-vn+1e-12)
                feats=[]
                with torch.no_grad():
                    for i in range(0,len(s),batch_size):
                        b=s[i:i+batch_size]; B,T,H,W,C=b.shape
                        if C==1: b=np.repeat(b,3,-1)
                        t=torch.tensor(b).permute(0,4,1,2,3).float()
                        if H!=112 or W!=112:
                            t=t.reshape(B*T,3,H,W)
                            t=F.interpolate(t,112,mode='bilinear',align_corners=False)
                            t=t.reshape(B,3,T,112,112)
                        t=(t-_mu)/_sd
                        feats.append(r3d(t.to(device)).squeeze(-1).squeeze(-1).squeeze(-1).cpu().numpy())
                return np.concatenate(feats)
            N=(len(pred_arr)//frames_per_seq)*frames_per_seq
            H,W,C=pred_arr.shape[1:]
            ps=pred_arr[:N].reshape(-1,frames_per_seq,H,W,C)
            gs=gt_arr[:N].reshape(-1,frames_per_seq,H,W,C)
            gf=r3d_feats(gs); pf=r3d_feats(ps)
            d=gf.mean(0)-pf.mean(0)
            cov=sqrtm(np.cov(gf,rowvar=False)@np.cov(pf,rowvar=False))
            if np.iscomplexobj(cov): cov=cov.real
            results['fvd_r3d18']=float(d@d+np.trace(
                np.cov(gf,rowvar=False)+np.cov(pf,rowvar=False)-2*cov))
        except ImportError as e:
            print(f'    跳过 FVD: {e}')

    all_results[name] = results

    # 保存单模型结果
    if m['type'] == 'fluid_vsr':
        with open(os.path.join(output_dir, 'results.json'), 'w') as f:
            json.dump(results, f, indent=2)

# ── 回填效率 JSON ─────────────────────────────────────────────────────────────
stats_dir = '/data/yc/Fluid_VSR/efficiency_stats'
_name_map = {'FNO': 'FNO2d', 'ReMD': 'REMG'}  # notebook name -> JSON name

def _extract_train_stats_from_log(model_dir):
    """从 train.log 提取训练时间等效率信息"""
    import glob, re as _re
    log_path = os.path.join(model_dir, 'train.log')
    if not os.path.exists(log_path):
        return {}
    with open(log_path) as f:
        lines = f.readlines()
    # 提取第一行和最后一个 Epoch 行的时间戳
    first_ts = last_ts = None
    total_epochs = 0
    time_per_epoch = None
    for line in lines:
        ts_m = _re.match(r'(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})', line)
        if ts_m and first_ts is None:
            first_ts = ts_m.group(1)
        if ts_m:
            last_ts = ts_m.group(1)
        ep_m = _re.search(r'Epoch (\d+).*Time: ([\d.]+)s', line)
        if ep_m:
            total_epochs = int(ep_m.group(1)) + 1
            time_per_epoch = float(ep_m.group(2))
    if first_ts and last_ts:
        from datetime import datetime
        fmt = '%Y-%m-%d %H:%M:%S'
        total_sec = (datetime.strptime(last_ts, fmt) - datetime.strptime(first_ts, fmt)).total_seconds()
    else:
        total_sec = None
    return {
        'total_epochs': total_epochs or None,
        'time_per_epoch_sec': time_per_epoch,
        'total_time_sec': total_sec,
        'total_iters': total_epochs * (time_per_epoch or 0) if total_epochs and time_per_epoch else None,
    }

for name, eff in all_efficiency.items():
    json_name = _name_map.get(name, name)
    json_path = os.path.join(stats_dir, f'{json_name}_{DATASET_NAME}.json')
    
    # 计算推理统计
    n_total_seqs = eff['n_seqs'] * (all_arrays[name]['pred'].shape[0] // all_arrays[name]['frames_per_seq'])
    
    if os.path.exists(json_path):
        with open(json_path) as f:
            stats = json.load(f)
    else:
        # 新建 JSON：从 model_dir 提取训练信息
        m_cfg = next((m for m in models if m['name'] == name), {})
        model_dir = m_cfg.get('model_dir', '')
        train_stats = _extract_train_stats_from_log(model_dir)
        # 从 forecaster 获取参数量和模型大小
        try:
            fc = get_forecaster(m_cfg)
            params = sum(p.numel() for p in fc.model.parameters())
            import tempfile, os as _os
            tmp = tempfile.mktemp(suffix='.pth')
            torch.save(fc.model.state_dict(), tmp)
            size_mb = round(_os.path.getsize(tmp) / 1024**2, 2)
            _os.remove(tmp)
        except Exception:
            params = None; size_mb = None
        
        # 从 config.yaml 获取 batch_size
        try:
            import yaml
            cfg = yaml.safe_load(open(os.path.join(model_dir, 'config.yaml')))
            batch_size_train = cfg.get('data', {}).get('train_batchsize', None)
        except Exception:
            batch_size_train = None
        
        stats = {
            'model_name': json_name,
            'dataset': DATASET_NAME,
            'params_total': params,
            'params_trainable': params,
            'model_size_mb': size_mb,
            'training': {
                'batch_size': batch_size_train,
                'batch_size_note': 'frame-level (not sequence-level)',
                'total_epochs': train_stats.get('total_epochs'),
                'total_iters': train_stats.get('total_iters'),
                'time_per_epoch_sec': train_stats.get('time_per_epoch_sec'),
                'time_per_iter_sec': None,
                'total_time_sec': train_stats.get('total_time_sec'),
                'peak_memory_mb': None,
            },
            'inference': {
                'total_sequences': None,
                'time_per_sequence_sec': None,
                'total_time_sec': None,
                'peak_memory_mb': None,
            }
        }
        print(f'  Created {json_path}')
    
    stats['inference']['total_sequences'] = n_total_seqs
    stats['inference']['time_per_sequence_sec'] = eff['infer_time_per_seq']
    stats['inference']['total_time_sec'] = eff['infer_time_total']
    stats['inference']['peak_memory_mb'] = eff['peak_memory_mb']
    with open(json_path, 'w') as f:
        json.dump(stats, f, indent=2)
    print(f'  Saved {json_path}')

print('\n✓ 全部完成.')

In [ ]:
# ── 对比表 ─────────────────────────────────────────────────────────────────────

_base_metrics = ['mse', 'rmse', 'psnr', 'ssim', 'relative_l2']
_vort_metrics = ['vorticity_error', 'enstrophy_error']
metric_order = _base_metrics + (_vort_metrics if include_vorticity else []) + ['psdd', 'fid', 'fvd_r3d18']

metric_fmt = {'mse':6, 'rmse':6, 'psnr':4, 'ssim':6,
              'relative_l2':6, 'vorticity_error':6, 'enstrophy_error':6,
              'fid':3, 'fvd_r3d18':3}
arrow = {'mse':'↓','rmse':'↓','psnr':'↑','ssim':'↑',
         'relative_l2':'↓','vorticity_error':'↓','enstrophy_error':'↓',
         'psdd':'↓','fid':'↓','fvd_r3d18':'↓'}

present = [k for k in metric_order if any(k in r for r in all_results.values())]

PSDD_WIDTH = 20
col_w = max(PSDD_WIDTH if 'psdd' in present else 12, max(len(n) for n in all_results))

header = f"{'Metric':<20}" + "".join(f"{n:>{col_w}}" for n in all_results)
print(header)
print('─' * len(header))
for k in present:
    row = f"{k+' '+arrow.get(k,''):<20}"
    for name in all_results:
        v = all_results[name].get(k, float('nan'))
        if k == 'psdd':
            row += f"{v:>{col_w}.11e}"
        else:
            fmt = metric_fmt.get(k, 4)
            row += f"{v:>{col_w}.{fmt}f}"
    print(row)
print('─' * len(header))

In [ ]:
# ── 可视化：每个模型单独一张大图 ─────────────────────────────────────────────
#   大图布局：4 行 × 5 列
#     Row 0: LR, Row 1: GT, Row 2: Pred, Row 3: |Pred - GT|

_all_err_vals = []
for _n, _arr in all_arrays.items():
    if _n == SDIFT_NAME:
        continue
    _fps   = _arr['frames_per_seq']
    _start = vis_seq_id * _fps
    _gt    = _arr['gt']  [_start:_start+_fps].squeeze(-1)
    _pred  = _arr['pred'][_start:_start+_fps].squeeze(-1)
    for fi in vis_frames:
        _all_err_vals.append(np.abs(_pred[fi] - _gt[fi]).ravel())

_non_sdift_err_vmax = float(np.percentile(np.concatenate(_all_err_vals), err_colorbar_percentile))
print(f'AbsError colorbar vmax (p{err_colorbar_percentile}): {_non_sdift_err_vmax:.6f}')

row_labels = ['LR', 'GT', 'Pred', 'Abs Error']
n_cols = len(vis_frames)
n_rows = 4

save_dir_vis = os.path.join(output_image_dir, 'error_images')
os.makedirs(save_dir_vis, exist_ok=True)

for model_name in all_arrays:
    arr      = all_arrays[model_name]
    fps      = arr['frames_per_seq']
    start    = vis_seq_id * fps

    lr_seq   = arr['lr']  [start:start+fps] if arr['lr'] is not None else None
    gt_seq   = arr['gt']  [start:start+fps].squeeze(-1)
    pred_seq = arr['pred'][start:start+fps].squeeze(-1)

    if lr_seq is not None:
        lr_seq = lr_seq.squeeze(-1)
        if lr_seq.max() <= 1.5 and gt_seq.max() > 2.0:
            lr_min, lr_max = gt_seq.min(), gt_seq.max()
            lr_seq = lr_seq * (lr_max - lr_min) + lr_min
        if lr_seq.shape[1:] != gt_seq.shape[1:]:
            lr_t = torch.tensor(lr_seq).unsqueeze(1).float()
            lr_t = F.interpolate(lr_t, size=gt_seq.shape[1:], mode='nearest')
            lr_seq = lr_t.squeeze(1).numpy()

    vmin_gp = float(gt_seq.min())
    vmax_gp = float(gt_seq.max())
    err_vmax = None if model_name == SDIFT_NAME else _non_sdift_err_vmax

    fig = plt.figure(figsize=(3 * n_cols + 1.5, 3 * n_rows + 0.5))
    gs  = gridspec.GridSpec(n_rows, n_cols + 1,
                            width_ratios=[1]*n_cols + [0.05],
                            hspace=0.08, wspace=0.05)
    fig.suptitle(f'{model_name}  (seq {vis_seq_id}, {DATASET_NAME})', fontsize=12, fontweight='bold')

    im_gp = im_err = None
    for ri in range(n_rows):
        for ci, fi in enumerate(vis_frames):
            ax = fig.add_subplot(gs[ri, ci])
            if ri == 0:
                if lr_seq is not None:
                    im = ax.imshow(lr_seq[fi], cmap='RdBu_r', vmin=vmin_gp, vmax=vmax_gp)
                    im_gp = im
                else:
                    ax.text(0.5, 0.5, 'No LR', ha='center', va='center', transform=ax.transAxes)
            elif ri == 1:
                im = ax.imshow(gt_seq[fi], cmap='RdBu_r', vmin=vmin_gp, vmax=vmax_gp)
                if im_gp is None: im_gp = im
            elif ri == 2:
                ax.imshow(pred_seq[fi], cmap='RdBu_r', vmin=vmin_gp, vmax=vmax_gp)
            else:
                err = np.abs(pred_seq[fi] - gt_seq[fi])
                im_err = ax.imshow(err, cmap='hot', vmin=0, vmax=err_vmax)
            ax.set_xticks([]); ax.set_yticks([])
            if ri == 0: ax.set_title(f't={fi}', fontsize=9)
            if ci == 0: ax.set_ylabel(row_labels[ri], fontsize=10, fontweight='bold',
                                      rotation=0, labelpad=46, va='center')

    cax_gp = fig.add_subplot(gs[0:3, n_cols])
    fig.colorbar(im_gp, cax=cax_gp, orientation='vertical').ax.set_ylabel('Physical value', fontsize=7)
    cax_err = fig.add_subplot(gs[3, n_cols])
    cb_label = '|Error|' if model_name == SDIFT_NAME else f'|Error| (p{err_colorbar_percentile})'
    fig.colorbar(im_err, cax=cax_err, orientation='vertical').ax.set_ylabel(cb_label, fontsize=7)

    save_path = os.path.join(save_dir_vis, f'{model_name}_seq{vis_seq_id}.png')
    fig.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f'Saved: {save_path}')
    plt.show()

In [ ]:
# ── 对比可视化：GT（第一行）+ 各非 SDIFT 模型 AbsError（后续行）──────────────
#   布局：(1 + N_non_sdift) 行 × 5 列，共用分位数 colorbar

non_sdift_models = [n for n in all_arrays if n != SDIFT_NAME]
n_compare_rows = 1 + len(non_sdift_models)
n_cols = len(vis_frames)

save_dir_compare = os.path.join(output_image_dir, 'comparison_images')
os.makedirs(save_dir_compare, exist_ok=True)

_ref = all_arrays[non_sdift_models[0]]
_fps = _ref['frames_per_seq']
_start = vis_seq_id * _fps
gt_ref = _ref['gt'][_start:_start+_fps].squeeze(-1)
vmin_gp = float(gt_ref.min())
vmax_gp = float(gt_ref.max())

fig = plt.figure(figsize=(3 * n_cols + 1.5, 3 * n_compare_rows + 0.5))
gs  = gridspec.GridSpec(n_compare_rows, n_cols + 1,
                        width_ratios=[1]*n_cols + [0.05],
                        hspace=0.1, wspace=0.05)
fig.suptitle(f'Comparison — {DATASET_NAME} seq {vis_seq_id}', fontsize=12, fontweight='bold')

im_gp = im_err = None

for ci, fi in enumerate(vis_frames):
    ax = fig.add_subplot(gs[0, ci])
    im = ax.imshow(gt_ref[fi], cmap='RdBu_r', vmin=vmin_gp, vmax=vmax_gp)
    im_gp = im
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f't={fi}', fontsize=9)
    if ci == 0: ax.set_ylabel('GT', fontsize=10, fontweight='bold',
                               rotation=0, labelpad=46, va='center')

for row_i, model_name in enumerate(non_sdift_models, start=1):
    arr      = all_arrays[model_name]
    fps      = arr['frames_per_seq']
    start    = vis_seq_id * fps
    pred_seq = arr['pred'][start:start+fps].squeeze(-1)
    gt_seq   = arr['gt']  [start:start+fps].squeeze(-1)
    for ci, fi in enumerate(vis_frames):
        ax = fig.add_subplot(gs[row_i, ci])
        err = np.abs(pred_seq[fi] - gt_seq[fi])
        im_err = ax.imshow(err, cmap='hot', vmin=0, vmax=_non_sdift_err_vmax)
        ax.set_xticks([]); ax.set_yticks([])
        if ci == 0: ax.set_ylabel(model_name, fontsize=9, fontweight='bold',
                                   rotation=0, labelpad=46, va='center')

cax_gp = fig.add_subplot(gs[0, n_cols])
fig.colorbar(im_gp, cax=cax_gp).ax.set_ylabel('Physical value', fontsize=7)
cax_err = fig.add_subplot(gs[1:, n_cols])
fig.colorbar(im_err, cax=cax_err).ax.set_ylabel(f'|Error| (p{err_colorbar_percentile})', fontsize=7)

save_path = os.path.join(save_dir_compare, f'comparison_seq{vis_seq_id}.png')
fig.savefig(save_path, dpi=150, bbox_inches='tight')
print(f'Saved: {save_path}')
plt.show()

In [ ]:
# ── 误差随时间分布 ────────────────────────────────────────────────────────────
#   图1：所有模型（含 SDIFT）
#   图2：排除 SDIFT
#   横轴：时间帧索引，纵轴：所有测试序列的平均绝对误差

save_dir_err_time = os.path.join(output_image_dir, 'error_distribution')
os.makedirs(save_dir_err_time, exist_ok=True)

error_over_time = {}
for model_name, arr in all_arrays.items():
    fps      = arr['frames_per_seq']
    n_frames = arr['pred'].shape[0]
    n_seqs   = n_frames // fps
    gt_all   = arr['gt']  [:n_seqs*fps].squeeze(-1).reshape(n_seqs, fps, -1)
    pred_all = arr['pred'][:n_seqs*fps].squeeze(-1).reshape(n_seqs, fps, -1)
    err_per_frame = np.abs(pred_all - gt_all).mean(axis=2).mean(axis=0)
    error_over_time[model_name] = err_per_frame

def _plot_error_time(models_to_plot, title, save_name):
    fig, ax = plt.subplots(figsize=(12, 6))
    for model_name in models_to_plot:
        err = error_over_time[model_name]
        ax.plot(range(len(err)), err, marker='o', markersize=3,
                label=model_name, linewidth=1.5)
    ax.set_xlabel('Frame index', fontsize=12)
    ax.set_ylabel('Mean absolute error', fontsize=12)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.legend(loc='best', fontsize=9)
    ax.grid(True, alpha=0.3)
    save_path = os.path.join(save_dir_err_time, save_name)
    fig.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f'Saved: {save_path}')
    plt.show()

# 图1：所有模型
_plot_error_time(
    list(all_arrays.keys()),
    f'Error over time — {DATASET_NAME} (all models)',
    'error_over_time_all.png'
)

# 图2：排除 SDIFT
_models_no_sdift = [n for n in all_arrays if n != SDIFT_NAME]
if _models_no_sdift:
    _plot_error_time(
        _models_no_sdift,
        f'Error over time — {DATASET_NAME} (excluding {SDIFT_NAME})',
        f'error_over_time_no_{SDIFT_NAME}.png'
    )